In [22]:
# Librerias usadas
import sqlite3
import ollama

### Usamos estas librerias para lo siguiente:

* sqlite3: Para realizar consultas sql a la base de datos de manera directa y mediante Python

* Ollama: Para cargar el modelo LLM y que corra localmente dentro del codigo

---
## Parte 1: Extracción con SQL

In [23]:
# Creamos la query 
query = """
select distinct reviews.review_id,  
published_date,  
reviewers.name as reviewer_name,
seat_types.name as seat_type,
traveller_types.name as traveller_type,
routes.origin, routes.destination,                -- De la Tabla Routes
aircraft.family as aircraft_family,
overall_rating,
review_ratings.score AS value_for_money,          -- De la Tabla review_ratings
cabin_staff_service.score AS cabin_staff_service, -- De la Tabla review_ratings
header, body
from reviews

-- Nombre review
JOIN reviewers 
    ON reviews.reviewer_id = reviewers.reviewer_id

-- Tipo de viaje
LEFT JOIN traveller_types 
    ON reviews.traveller_type_id = traveller_types.traveller_type_id

-- Relación seat_types
JOIN seat_types 
    ON reviews.seat_type_id = seat_types.seat_type_id

-- Rutas
LEFT JOIN routes
    ON reviews.route_id = routes.route_id

-- Conexion a Tabla intermedia Review_aircraft
LEFT JOIN review_aircraft
    ON reviews.review_id = review_aircraft.review_id
    
-- Aircraft    
LEFT JOIN aircraft
    ON review_aircraft.aircraft_id = aircraft.aircraft_id

-- Ratings de Value for Money
JOIN review_ratings 
    ON reviews.review_id = review_ratings.review_id
    AND review_ratings.category_id = 4

-- Ratings de Cabin Staff Service
 LEFT JOIN review_ratings AS cabin_staff_service
    ON reviews.review_id = cabin_staff_service.review_id 
    AND cabin_staff_service.category_id = 2

-- Pasajero no recomienda la aerolinea
where review_ratings.score <= 2 
-- Pasajero esta verificado
and verified = 1
-- Pasajero que no recomienda la aerolinea 
and recommended = 0 
-- Ratings generales menor o iguales a 2
and overall_rating <=2

-- reviews con fecha
and published_date BETWEEN '2023-05-01' and '2023-10-31'

-- Economy Class y Premium Economy
and (reviews.seat_type_id = 1 or reviews.seat_type_id = 3)

order by published_date DESC;
"""

In [26]:
# Ejecutamos la query y la guardamos en un archivo .txt
with sqlite3.connect("airline_reviews.db") as conexion:
    cursor = conexion.cursor()
    cursor.execute(query)
    resultados = cursor.fetchall()

    # Creamos un archivo en modo escritura ('w')
    with open('resultados_reviews.txt', 'w', encoding='utf-8') as archivo:

        for fila in resultados:
            # Convertimos cada elemento de la tupla a string separandolo por comas
            linea = ",".join(str(campo) for campo in fila)
            
            # Escribimos la línea agregando el salto de línea al final
            archivo.write(linea + "\n")
    cursor.close()
print("Los resultados se han guardado correctamente en 'resultados_reviews.txt'.")

Los resultados se han guardado correctamente en 'resultados_reviews.txt'.


## Parte 2: Clasificación con un LLM 

In [ ]:
# Abrimos el documento recien creado
with open("resultados_reviews.txt", "r", encoding="utf-8") as f:
    resenas = [linea.strip() for linea in f if linea.strip()]

# Creamos distintos Diccionarios para almacenar las iteraciones

clasificaciones = []       # Guarda el texto de la reseña, la categoria y el area de mejora.
conteo_categorias = {}     # Para contar la cantidad de veces que se repite la categoria

# Bucle para pedirle al LLM que clasifica la reseña una por una
for resena in resenas:
    # Llamada directa a Ollama
    response = ollama.chat(
        model="qwen2.5:1.5b",         # Aqui se cambia el modelo, en caso de tener otro
        messages=[{
            'role': 'user', 
            'content': f"Clasifica esta reseña junto con el formato adjunto:\n{resena}" # Rol en el que responde el modelo
        }],
        format={
            "type": "object",
            "properties": {
                "categoria_problema": {
                    "type": "string",
                    "enum": [
                        "Gestión Operativa de Vuelos y Conexiones",
                        "Check-in, Embarque y Canales Digitales",
                        "Servicio al Cliente y Trato del Personal",
                        "Experiencia y Confort a Bordo",
                        "Equipaje e Infraestructura de la Terminal"
                    ]
                },
                "area_mejora": {
                    "type": "string",
                    "enum": [
                        "operations",
                        "cabin_crew",
                        "airport_service",
                        "onboard_product",
                        "customer_service",
                        "commercial_policy"
                    ]
                }
            },
            "required": ["categoria_problema", "area_mejora"]
        },
        options={'temperature': 0.0}
    )

    # La respuesta directa de ollama
    resultado_texto = response.message.content
    
    categoria = resultado_texto.split('"categoria_problema": "')[1].split('"')[0]

    # Conteo si la categoría existe suma 1, si no, inicia una nueva
    if categoria in conteo_categorias:
        conteo_categorias[categoria] += 1
    else:
        conteo_categorias[categoria] = 1

    # Extraer los datos
    categoria = resultado_texto.split('"categoria_problema": "')[1].split(
        '"'
    )[0]
    area = resultado_texto.split('"area_mejora": "')[1].split('"')[0]

    # Guardamos en la lista estructurada
    clasificaciones.append(
        {"texto": resena, "categoria": categoria, "area": area}
    )

# Mostramos los resultados tras correr el modelo
print("-" * 50)
print("Conteo por categoría")
print("-" * 50)

for categoria, cantidad in conteo_categorias.items():
    print(f"- {categoria}: {cantidad} reseñas")

print(f"\nTotal general procesado: {sum(conteo_categorias.values())} reseñas")

RESUMEN FINAL DE CONTABILIZACIÓN POR CATEGORÍA
- Servicio al Cliente y Trato del Personal: 50 reseñas
- Check-in, Embarque y Canales Digitales: 1 reseñas

Total general procesado: 51 reseñas.


## Parte 3: Persistencia y resumen

In [ ]:
# Nos conectamos a la base de datos para crear una nueva tabla con nuestras categorias
with sqlite3.connect("airline_reviews.db") as conexion:
    cursor = conexion.cursor()
# Creamos una tabla de clasificación para luego conectarla a la base de datos original
    cursor.execute(
        """
    CREATE TABLE IF NOT EXISTS clasification (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        review_id INTEGER NOT NULL,
        categoria_problema TEXT NOT NULL,
        area_mejora TEXT NOT NULL,
        FOREIGN KEY (review_id) REFERENCES reviews(id)
    )
    """
    )

    # Creamos otro bucle para procesar cada reseña e insertarla
    for i, item in enumerate(clasificaciones, 1):
        cursor.execute(
            """
            INSERT INTO clasification (review_id, categoria_problema, area_mejora)
            VALUES (?, ?, ?)
        """,
            (i, item["categoria"], item["area"]),
        )


In [ ]:
# Definimos una query para unir nuestra tabla a las demas
query = """
SELECT 
    clasification.categoria_problema,
    COUNT(reviews.review_id) AS cantidad_reviews,
    ROUND(AVG(reviews.overall_rating), 2) AS rating_promedio,
    (
        SELECT clasification_sub.area_mejora 
        FROM clasification AS clasification_sub
        WHERE clasification_sub.categoria_problema = clasification.categoria_problema
        GROUP BY clasification_sub.area_mejora
        ORDER BY COUNT(*) DESC
        LIMIT 1
    ) AS area_mejora_mas_frecuente
FROM reviews
JOIN clasification ON reviews.review_id = clasification.review_id
GROUP BY clasification.categoria_problema
ORDER BY cantidad_reviews DESC;
"""

# Nos conectamos y ejecutamos la consulta en SQLite3
with sqlite3.connect("airline_reviews.db") as conexion:
    cursor = conexion.cursor()
    cursor.execute(query)
    resultados = cursor.fetchall()

    # Finalmente mostramos un reporte en la consola
    print(
        f"{'Categoría Temática':<45} | {'Reviews':<8} | {'Rating Prom.':<12} | {'Área Mejora Más Frecuente'}"
    )
    print("-" * 100)

    for fila in resultados:
        categoria = fila[0]
        cantidad = fila[1]
        rating = fila[2]
        area_frecuente = fila[3]

        print(
            f"{categoria:<45} | {cantidad:<8} | {rating:<12} | {area_frecuente}"
        )
    cursor.close()

Categoría Temática                            | Reviews  | Rating Prom. | Área Mejora Más Frecuente
----------------------------------------------------------------------------------------------------
Servicio al Cliente y Trato del Personal      | 50       | 3.0          | airport_service
Check-in, Embarque y Canales Digitales        | 1        | 1.0          | airport_service
